# 03 - Agent 架构模式深度解析

本notebook深入介绍不同的Agent架构模式、实现原理和使用场景。

## 学习目标
- 理解Agent的核心组件和理论基础
- 掌握ReAct、ToolCalling、PlanAndExecute模式
- 理解Agent的运行循环和状态管理
- 学会调试和优化Agent行为
- 掌握多Agent协作模式

## 1. 理论基础

### 什么是AI Agent？

AI Agent是能够自主感知环境、推理决策并执行行动的系统：

```
┌─────────────────────────────────────────────────────────┐
│                    AI Agent 架构                         │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  ┌─────────┐    ┌─────────┐    ┌─────────┐             │
│  │感知 Perception│  │推理 Reasoning │  │行动 Action │   │
│  │         │    │         │    │         │             │
│  │- 输入   │    │- 规划   │    │- 工具   │             │
│  │- 上下文 │    │- 决策   │    │- 输出   │             │
│  └────┬────┘    └────┬────┘    └────┬────┘             │
│       │              │              │                   │
│       └──────────────┼──────────────┘                   │
│                      │                                  │
│              ┌───────┴────────┐                         │
│              │   Memory 记忆  │                         │
│              │                │                         │
│              │ - 对话历史     │                         │
│              │ - 知识库       │                         │
│              └────────────────┘                         │
└─────────────────────────────────────────────────────────┘
``` 

### 核心论文

1. **ReAct** (Yao et al., 2022): Reasoning + Acting协同
2. **Toolformer** (Schick et al., 2023): 自主学习工具使用
3. **Reflexion** (Shinn et al., 2023): 自我反思改进
4. **COT** (Wei et al., 2022): 思维链推理
5. **ToT** (Yao et al., 2023): 思维树探索

In [ ]:
import sys
sys.path.insert(0, '../src')

from agent import (
    AgentConfig, AgentState, AgentAction, AgentFinish,
    ReActAgent, ToolCallingAgent, PlanAndExecuteAgent
)
from tools import CalculatorTool, SearchTool, ToolRegistry, DateTimeTool
from memory import BufferMemory, WindowMemory
import json
from typing import List, Dict, Any, Optional

## 2. Agent 配置详解

### AgentConfig 参数说明

In [ ]:
# 创建不同的Agent配置
configs = {
    "快速模式": AgentConfig(
        max_iterations=5,
        max_execution_time=30.0,
        early_stopping=True,
        return_intermediate_steps=False
    ),
    "标准模式": AgentConfig(
        max_iterations=10,
        max_execution_time=120.0,
        early_stopping=True,
        return_intermediate_steps=True
    ),
    "调试模式": AgentConfig(
        max_iterations=15,
        max_execution_time=300.0,
        early_stopping=False,
        return_intermediate_steps=True,
        handle_parsing_errors=True,
        verbose=True
    )
}

print("=== Agent 配置对比 ===")
for name, config in configs.items():
    print(f"\n{name}:")
    print(f"  max_iterations: {config.max_iterations}")
    print(f"  max_execution_time: {config.max_execution_time}s")
    print(f"  early_stopping: {config.early_stopping}")
    print(f"  return_intermediate_steps: {config.return_intermediate_steps}")

## 3. ReAct Agent 深度解析

ReAct (Reasoning + Acting) 是最经典的Agent模式之一。

In [ ]:
# 创建ReAct Agent
def mock_llm_with_react(messages):
    """
    模拟LLM返回ReAct格式的响应
    格式: Thought -> Action -> Action Input
    """
    return """
Thought: 我需要计算这个数学表达式
Action: calculator
Action Input: {"expression": "2 + 3 * 4"}
    """.strip()

react_agent = ReActAgent(
    tools=[CalculatorTool(), SearchTool(), DateTimeTool()],
    config=AgentConfig(max_iterations=5),
    llm=mock_llm_with_react
)

print("=== ReAct Agent 信息 ===")
print(f"Agent类型: {type(react_agent).__name__}")
print(f"可用工具: {react_agent.tool_registry.list_tools()}")
print(f"当前状态: {react_agent.state}")

In [ ]:
# 测试ReAct响应解析
test_responses = [
    # 完整的ReAct格式
    """Thought: 我需要搜索信息
Action: search
Action Input: {"query": "Python"}""",
    
    # 只有思考
    "Thought: 让我思考一下这个问题的答案",
    
    # 最终答案
    "Final Answer: 答案是42",
    
    # JSON格式的Action Input
    "Thought: 需要计算
Action: calculator
Action Input: {\"expression\": \"1 + 1\"}",
]

print("=== ReAct 响应解析测试 ===")
for i, response in enumerate(test_responses, 1):
    print(f"\n--- 测试 {i} ---")
    print(f"输入: {response[:50]}...")
    
    try:
        parsed = react_agent._parse_response(response)
        print(f"解析类型: {type(parsed).__name__}")
        if isinstance(parsed, AgentAction):
            print(f"  工具: {parsed.tool}")
            print(f"  参数: {parsed.tool_input}")
        elif isinstance(parsed, AgentFinish):
            print(f"  最终答案: {parsed.output}")
    except Exception as e:
        print(f"  解析错误: {e}")

In [ ]:
# ReAct 执行流程示例
def react_example_flow():
    """
    演示完整的ReAct循环
    """
    print("\n=== ReAct 执行流程演示 ===")
    
    # 模拟多轮对话
    steps = [
        ("Thought: 我需要计算", "Action: calculator", "{\"expression\": \"2*3\"}"),
        ("Observation: 6", "Thought: 现在加4", "Action: calculator", "{\"expression\": \"6+4\"}"),
        ("Observation: 10", "Thought: 我有答案了", "Final Answer: 10"),
    ]
    
    for step_num, step in enumerate(steps, 1):
        print(f"\n[步骤 {step_num}]")
        for item in step:
            print(f"  {item}")

react_example_flow()

## 4. ToolCalling Agent

使用结构化函数调用，适合支持Function Calling的模型。

In [ ]:
def mock_llm_tool_calling(messages):
    """
    模拟Function Calling格式的LLM响应
    """
    return '{"tool": "calculator", "arguments": {"expression": "sqrt(144)"}}'

tc_agent = ToolCallingAgent(
    tools=[CalculatorTool(), DateTimeTool()],
    llm=mock_llm_tool_calling
)

print("=== ToolCalling Agent ===")
print(f"可用工具: {tc_agent.tool_registry.list_tools()}")

In [ ]:
# 测试Function Calling解析
tool_calling_examples = [
    # 标准格式
    '{"tool": "calculator", "arguments": {"expression": "2+2"}}',
    
    # 嵌套JSON
    '{"tool": "search", "arguments": {"query": "Python教程", "limit": 5}}',
    
    # 无工具调用
    '直接回答: Python是一种编程语言',
    
    # 多个参数
    '{"tool": "datetime", "arguments": {"action": "add", "days": 7, "unit": "days"}}',
]

print("=== Function Calling 解析测试 ===")
for example in tool_calling_examples:
    print(f"\n输入: {example[:60]}...")
    try:
        parsed = tc_agent._parse_response(example)
        if isinstance(parsed, AgentAction):
            print(f"  -> 工具: {parsed.tool}")
            print(f"  -> 参数: {parsed.tool_input}")
        else:
            print(f"  -> 非工具调用: {parsed.output if hasattr(parsed, 'output') else parsed}")
    except Exception as e:
        print(f"  -> 解析失败: {type(e).__name__}: {e}")

## 5. PlanAndExecute Agent

先规划再执行的Agent，适合复杂多步骤任务。

In [ ]:
pe_agent = PlanAndExecuteAgent(
    tools=[CalculatorTool(), SearchTool(), DateTimeTool()],
    config=AgentConfig(max_iterations=10)
)

print("=== PlanAndExecute Agent ===")
print(f"可用工具: {pe_agent.tool_registry.list_tools()}")

In [ ]:
# 测试步骤到工具的映射
test_steps = [
    "计算 123 + 456",
    "搜索 Python 教程",
    "查询当前日期",
    "计算 7 天后的日期",
    "写一份报告",  # 没有对应工具
    "分析数据趋势",  # 没有对应工具
]

print("=== 步骤到工具的映射 ===")
for step in test_steps:
    tool = pe_agent._select_tool_for_step(step)
    if tool:
        print(f"\n步骤: {step}")
        print(f"  -> 工具: {tool}")
    else:
        print(f"\n步骤: {step}")
        print(f"  -> 无匹配工具")

In [ ]:
# 模拟规划阶段
task = "帮我计算从2024年1月1日到现在有多少天，然后搜索2024年的重要事件"

print("\n=== 规划阶段示例 ===")
print(f"任务: {task}")
print("\n可能的计划:")
plan = [
    "1. 获取当前日期",
    "2. 计算日期差",
    "3. 搜索2024年重要事件",
    "4. 综合信息生成报告",
]
for step in plan:
    print(f"  {step}")

print("\n工具分配:")
for step in plan:
    tool = pe_agent._select_tool_for_step(step)
    print(f"  {step}: {tool or '无需工具'}")

## 6. Agent 状态管理

Agent在运行过程中会经历不同的状态。

In [ ]:
print("=== Agent 状态枚举 ===")
print("AgentState 的可能值:")
for state in AgentState:
    print(f"  - {state.value}")

In [ ]:
# AgentAction 和 AgentFinish
print("=== AgentAction 示例 ===")
action = AgentAction(
    tool="calculator",
    tool_input={"expression": "2 * 3"},
    log="Thinking: Need to calculate"
)
print(f"工具: {action.tool}")
print(f"参数: {action.tool_input}")
print(f"日志: {action.log}")

print("\n=== AgentFinish 示例 ===")
finish = AgentFinish(
    output="计算结果是6",
    log="Task completed"
)
print(f"输出: {finish.output}")
print(f"日志: {finish.log}")

## 7. Agent 运行循环详解

理解Agent的核心执行流程。

In [ ]:
def simulate_agent_run(agent, input_text: str):
    """
    模拟Agent的运行循环
    """
    print(f"\n=== Agent 运行模拟 ===")
    print(f"输入: {input_text}")
    print(f"初始状态: {agent.state.value}")
    
    iteration = 0
    max_iterations = 3
    intermediate_steps = []
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- 迭代 {iteration} ---")
        
        # 1. 规划阶段
        print("[规划] 思考下一步动作...")
        
        # 模拟LLM响应
        if iteration == 1:
            response = "Thought: 需要计算\nAction: calculator\nAction Input: {\"expression\": \"10+20\"}"
        elif iteration == 2:
            response = "Observation: 30\nThought: 需要再加5\nAction: calculator\nAction Input: {\"expression\": \"30+5\"}"
        else:
            response = "Final Answer: 最终结果是35"
        
        # 2. 解析响应
        parsed = agent._parse_response(response)
        print(f"[解析] 结果类型: {type(parsed).__name__}")
        
        # 3. 检查是否完成
        if isinstance(parsed, AgentFinish):
            print(f"[完成] {parsed.output}")
            return parsed.output
        
        # 4. 执行动作
        if isinstance(parsed, AgentAction):
            print(f"[执行] 调用工具: {parsed.tool}")
            print(f"       参数: {parsed.tool_input}")
            
            tool = agent.tool_registry.get_tool(parsed.tool)
            if tool:
                result = tool.run(**parsed.tool_input)
                observation = result.output
                print(f"[结果] {observation}")
                intermediate_steps.append((parsed, observation))
    
    return "达到最大迭代次数"

# 运行模拟
agent = ReActAgent(
    tools=[CalculatorTool()],
    config=AgentConfig(max_iterations=5)
)
result = simulate_agent_run(agent, "计算 10 + 20 + 5")
print(f"\n最终结果: {result}")

## 8. 工具选择策略

In [ ]:
# 智能工具选择
registry = ToolRegistry()
registry.register(CalculatorTool())
registry.register(SearchTool())
registry.register(DateTimeTool())

def suggest_tool(query: str, registry: ToolRegistry) -> Optional[str]:
    """
    根据查询建议合适的工具
    """
    query_lower = query.lower()
    
    # 计算相关关键词
    calc_keywords = ['计算', '加', '减', '乘', '除', 'sqrt', 'sin', 'cos', '数学', '等于']
    # 搜索相关关键词
    search_keywords = ['搜索', '查询', '找', '百度', 'google', '信息']
    # 日期相关关键词
    date_keywords = ['日期', '时间', '几号', '今天', '明天', '天后', '天前']
    
    scores = {}
    for keyword in calc_keywords:
        if keyword in query_lower:
            scores['calculator'] = scores.get('calculator', 0) + 1
    
    for keyword in search_keywords:
        if keyword in query_lower:
            scores['search'] = scores.get('search', 0) + 1
    
    for keyword in date_keywords:
        if keyword in query_lower:
            scores['datetime'] = scores.get('datetime', 0) + 1
    
    if scores:
        return max(scores.items(), key=lambda x: x[1])[0]
    return None

# 测试工具选择
test_queries = [
    "帮我计算 2 + 3",
    "搜索 Python 教程",
    "今天几号？",
    "100 除以 5 等于多少",
    "查询机器学习资料",
    "7天后是几号",
]

print("=== 智能工具选择 ===")
for query in test_queries:
    tool = suggest_tool(query, registry)
    print(f"\n查询: {query}")
    print(f"建议工具: {tool or '无需工具'}")

## 9. 错误处理和恢复

In [ ]:
# 测试错误处理
class RobustAgent:
    """
    带错误处理的Agent示例
    """
    
    def __init__(self, tools, config):
        self.tool_registry = ToolRegistry()
        for tool in tools:
            self.tool_registry.register(tool)
        self.config = config
    
    def execute_with_retry(self, action: AgentAction, max_retries: int = 2):
        """
        执行动作并支持重试
        """
        for attempt in range(max_retries + 1):
            try:
                tool = self.tool_registry.get_tool(action.tool)
                if not tool:
                    return f"错误: 工具 '{action.tool}' 不存在"
                
                result = tool.run(**action.tool_input)
                if result.success:
                    return result.output
                else:
                    return f"错误: {result.error}"
                    
            except Exception as e:
                if attempt < max_retries:
                    continue
                return f"执行失败: {str(e)}"
        
        return "达到最大重试次数"

# 创建RobustAgent并测试
robust_agent = RobustAgent(
    tools=[CalculatorTool()],
    config=AgentConfig(max_iterations=5)
)

# 测试各种场景
test_actions = [
    AgentAction(tool="calculator", tool_input={"expression": "2+2"}, log=""),
    AgentAction(tool="calculator", tool_input={"expression": "1/0"}, log=""),  # 会出错
    AgentAction(tool="unknown_tool", tool_input={}, log=""),  # 工具不存在
]

print("=== 错误处理测试 ===")
for action in test_actions:
    result = robust_agent.execute_with_retry(action)
    print(f"\n工具: {action.tool}")
    print(f"结果: {result}")

## 10. 中间步骤追踪

In [ ]:
# 记录和展示中间步骤
class StepTracker:
    """
    Agent执行步骤追踪器
    """
    
    def __init__(self):
        self.steps = []
    
    def add_step(self, action: AgentAction, observation: str):
        self.steps.append({
            "iteration": len(self.steps) + 1,
            "tool": action.tool,
            "input": action.tool_input,
            "output": observation,
            "log": action.log
        })
    
    def display(self):
        print("\n=== Agent 执行步骤 ===")
        for step in self.steps:
            print(f"\n[步骤 {step['iteration']}] 工具: {step['tool']}")
            print(f"  输入: {step['input']}")
            print(f"  输出: {step['output'][:80]}..." if len(str(step['output'])) > 80 else f"  输出: {step['output']}")
    
    def to_dict(self):
        return self.steps

# 使用步骤追踪器
tracker = StepTracker()

# 模拟添加步骤
tracker.add_step(
    AgentAction(tool="calculator", tool_input={"expression": "10*5"}, log="计算10*5"),
    "50"
)
tracker.add_step(
    AgentAction(tool="calculator", tool_input={"expression": "50+20"}, log="加20"),
    "70"
)

tracker.display()

## 11. 多Agent协作

In [ ]:
class MultiAgentSystem:
    """
    多Agent协作系统
    """
    
    def __init__(self):
        self.agents = {}
        self.communication_log = []
    
    def add_agent(self, name: str, agent):
        self.agents[name] = agent
    
    def route_task(self, task: str) -> str:
        """
        根据任务路由到合适的Agent
        """
        task_lower = task.lower()
        
        # 简单路由逻辑
        if any(kw in task_lower for kw in ['计算', '数学', '等于']):
            return 'math_agent'
        elif any(kw in task_lower for kw in ['搜索', '查询', '信息']):
            return 'search_agent'
        elif any(kw in task_lower for kw in ['日期', '时间']):
            return 'date_agent'
        else:
            return 'general_agent'
    
    def execute(self, task: str, agent_name: str = None) -> str:
        """
        执行任务
        """
        if agent_name is None:
            agent_name = self.route_task(task)
        
        agent = self.agents.get(agent_name)
        if not agent:
            return f"错误: Agent '{agent_name}' 不存在"

In [ ]:
        
        self.communication_log.append({
            "task": task,
            "routed_to": agent_name,
            "timestamp": len(self.communication_log)
        })
        
        return f"[{agent_name}] 处理任务: {task}"
    
    def show_routing_history(self):
        print("\n=== 任务路由历史 ===")
        for log in self.communication_log:
            print(f"任务: {log['task'][:40]}... -> {log['routed_to']}")

# 创建多Agent系统
multi_agent = MultiAgentSystem()

# 添加专门的Agent
multi_agent.add_agent('math_agent', None)  # 实际应该是ReActAgent实例
multi_agent.add_agent('search_agent', None)
multi_agent.add_agent('date_agent', None)
multi_agent.add_agent('general_agent', None)

# 测试任务路由
tasks = [
    "帮我计算 100 + 200",
    "搜索最新的AI新闻",
    "今天是什么日期",
    "给我介绍一下Python",
]

print("=== 多Agent协作演示 ===")
for task in tasks:
    result = multi_agent.execute(task)
    print(f"\n{result}")

multi_agent.show_routing_history()

## 12. Agent 性能对比

In [ ]:
# Agent模式对比
agent_comparison = [
    {
        "模式": "ReAct",
        "优点": [
            "推理过程透明",
            "易于调试",
            "适合复杂推理"
        ],
        "缺点": [
            "Token消耗较大",
            "解析可能不稳定"
        ],
        "适用场景": "需要解释推理过程的任务"
    },
    {
        "模式": "ToolCalling",
        "优点": [
            "结构化输出",
            "解析可靠性高",
            "Token效率高"
        ],
        "缺点": [
            "依赖模型能力",
            "推理过程不透明"
        ],

In [ ]:
        "适用场景": "支持Function Calling的模型"
    },
    {
        "模式": "PlanAndExecute",
        "优点": [
            "适合复杂任务",
            "可以调整计划",
            "步骤清晰"
        ],
        "缺点": [
            "规划可能不准",
            "需要多次迭代"
        ],
        "适用场景": "多步骤复杂任务"
    }
]

print("\n=== Agent 模式对比 ===")
for agent_type in agent_comparison:
    print(f"\n### {agent_type['模式']} Agent")
    print(f"优点:")
    for pro in agent_type['优点']:
        print(f"  + {pro}")
    print(f"缺点:")
    for con in agent_type['缺点']:
        print(f"  - {con}")
    print(f"适用场景: {agent_type['适用场景']}")

## 总结

### Agent 设计要点

1. **选择合适的模式**
   - 简单任务: ToolCalling
   - 复杂推理: ReAct
   - 多步任务: PlanAndExecute

2. **配置优化**
   - max_iterations: 防止无限循环
   - max_execution_time: 控制执行时间
   - early_stopping: 提前结束策略
   - handle_parsing_errors: 错误处理

3. **调试技巧**
   - 启用 return_intermediate_steps
   - 使用 StepTracker 追踪执行
   - 打印中间结果

4. **性能优化**
   - 合理使用记忆
   - 批量执行工具
   - 缓存重复结果

### 三种模式对比

| 特性 | ReAct | ToolCalling | PlanAndExecute |
|------|-------|-------------|----------------|
| 推理透明度 | 高 | 低 | 中 |
| Token效率 | 低 | 高 | 中 |
| 解析可靠性 | 中 | 高 | 中 |
| 复杂任务能力 | 强 | 中 | 最强 |
| 调试难度 | 低 | 中 | 高 |